# Databricks Vector Search — Concepts, Grounded in Code

Databricks Vector Search (recently renamed **AI Search**) is a managed vector database built into
the Databricks platform. This notebook doesn't call a live Databricks endpoint (this environment
has no workspace to connect to) — instead it takes the concepts documented at
[docs.databricks.com/aws/en/ai-search/ai-search](https://docs.databricks.com/aws/en/ai-search/ai-search),
[databricks.com/blog/what-is-vector-database](https://www.databricks.com/blog/what-is-vector-database),
and [databricks.com/blog/vector-search](https://www.databricks.com/blog/vector-search), and makes
the *mechanics* concrete by implementing Databricks' own documented formulas against this
project's synthetic vectors and existing index code. See the companion doc,
[`../docs/Similarity_Search_Methods/08_Databricks_Vector_Search.md`](../docs/Similarity_Search_Methods/08_Databricks_Vector_Search.md),
for the full write-up this notebook accompanies.

## 1. What it is, architecturally

Three stages, same as every method in this project's taxonomy
([`../docs/Similarity_Search_Methods/01_Vector_Based_Methods_for_Similarity_Search.md`](../docs/Similarity_Search_Methods/01_Vector_Based_Methods_for_Similarity_Search.md)),
just managed end-to-end by the platform instead of assembled by hand:

1. **Embedding creation** — Databricks-hosted or self-managed models turn rows into vectors.
2. **Index construction** — vectors are stored in a structure optimized for similarity search.
3. **Query matching** — an incoming query is embedded and matched via **approximate nearest
   neighbor (ANN)** search, not exhaustive brute force — the same exact/approximate trade-off
   covered in this project's [benchmark](../docs/Similarity_Search_Methods/06_Choosing_and_Benchmarking.md).

The ANN algorithm underneath is documented as **HNSW** — the exact graph-based method implemented
in [`graph_ann.py`](graph_ann.py) and explained in
[`04_Graph_Based_Methods.md`](../docs/Similarity_Search_Methods/04_Graph_Based_Methods.md). Nothing
Databricks-specific about the algorithm itself — the product is the managed infrastructure *around*
an ANN index (sync pipelines, governance, serving), not a new search algorithm.

## 2. The documented similarity score formula

Databricks' docs state the index uses **L2 (Euclidean) distance** internally, converted to a
similarity score via:

```
similarity_score = 1 / (1 + distance²)
```

and that *"when vectors are normalized, L2-distance ranking is the same as cosine-similarity
ranking"* — i.e. you can request cosine-similarity semantics without the index actually computing
cosine similarity, as long as you normalize your embeddings first. Let's verify that claim
directly, using this project's own [`metrics.py`](metrics.py) and synthetic dataset.

In [1]:
import numpy as np

from data import make_dataset, make_queries
from metrics import cosine_similarity, euclidean_distance

vectors, _, centers = make_dataset(n_vectors=5000, dim=64)
queries = make_queries(centers, n_queries=20)


def databricks_similarity_score(query, vectors):
    """similarity_score = 1 / (1 + distance^2), as documented for Databricks Vector Search."""
    distance = euclidean_distance(query, vectors)
    return 1 / (1 + distance**2)


agreements = []
for q in queries:
    databricks_rank = np.argsort(-databricks_similarity_score(q, vectors))[:10]
    cosine_rank = np.argsort(-cosine_similarity(q, vectors))[:10]
    agreements.append(list(databricks_rank) == list(cosine_rank))

print(f"Identical top-10 ranking (L2 score vs. cosine) on {len(queries)} queries: {sum(agreements)}/{len(queries)}")

Identical top-10 ranking (L2 score vs. cosine) on 20 queries: 20/20


Since this project's `make_dataset()` already unit-normalizes every vector (see
[`data.py`](data.py)), the ranking matches exactly — confirming the documented claim: **for
unit-normalized embeddings, Databricks' L2-based `similarity_score` and cosine similarity produce
the identical ranking.** This is the same normalization fact covered generally in
[`01_Vector_Based_Methods_for_Similarity_Search.md §3`](../docs/Similarity_Search_Methods/01_Vector_Based_Methods_for_Similarity_Search.md#3-similarity-metrics-revisited)
— Databricks isn't doing anything mathematically novel here, just picking one metric (L2) as the
index's native operation and documenting how to get cosine-equivalent results from it.

## 3. Index types: how embeddings get into the index

| Index type | Who computes embeddings | Sync | Analogy in this project |
|---|---|---|---|
| **Delta Sync Index (Databricks-managed)** | Platform, using a specified model serving endpoint | Automatic, on source Delta table change | `data.py` + any of `brute_force.py`/`graph_ann.py`/etc. rebuilt on new data |
| **Delta Sync Index (self-managed)** | You, pre-computed into a Delta table column | Automatic sync of your precomputed vectors | Precomputing `vectors = model.encode(texts)` before calling `HNSWIndex(vectors)` |
| **Direct Vector Access Index** | You, via API calls | Manual — you push updates yourself | Rebuilding `HNSWIndex(vectors)` by hand whenever `vectors` changes |
| **Full-text search (storage-optimized, Beta)** | N/A — keyword-only, BM25 scoring | N/A | No analog in this project — pure keyword search, no embeddings at all |

The first three are all backed by the same underlying HNSW index — they differ only in *who is
responsible for producing embeddings* and *how freshness is maintained*, not in the search
algorithm itself.

## 4. Hybrid search: vector + keyword, fused with Reciprocal Rank Fusion (RRF)

Databricks Vector Search can combine **dense vector search** (semantic, cosine/L2-based) with
**sparse keyword search** (BM25, exact-term matching — good for product codes, IDs, acronyms that
embeddings often blur together). The two rankings are merged using **Reciprocal Rank Fusion**
with `rrf_param=60`, documented as:

```
RRF_score(doc) = Σ over each ranked list :  1 / (k + rank_in_that_list)      where k = 60
```

A document ranked highly in *either* list contributes a large term; a document appearing in *both*
lists accumulates both contributions — which is exactly why RRF is a good fusion strategy: it
doesn't require the two rankers' raw scores to be on a comparable scale (BM25 scores and cosine
similarities aren't), only their *rank positions*, which are always comparable.

In [2]:
def reciprocal_rank_fusion(rank_lists, k=60):
    """Databricks' documented fusion strategy: rrf_param=60, applied to a list of ranked-id lists."""
    scores = {}
    for ranks in rank_lists:
        for position, doc_id in enumerate(ranks):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + position + 1)
    return sorted(scores.items(), key=lambda item: -item[1])


# A toy stand-in for two independent rankers over the same 10 documents: one from vector search
# (e.g. HNSWIndex.search), one from a keyword/BM25 ranker (not implemented in this project —
# out of scope, this project is vector-methods-only). Doc 3 and doc 7 rank well in both signals.
vector_search_ranking = [3, 7, 1, 9, 2, 5, 8, 0, 6, 4]
keyword_search_ranking = [1, 3, 5, 7, 8, 2, 9, 6, 0, 4]

fused = reciprocal_rank_fusion([vector_search_ranking, keyword_search_ranking])
print("doc_id  rrf_score")
for doc_id, score in fused[:5]:
    print(f"{doc_id:6d}  {score:.5f}")

doc_id  rrf_score
     3  0.03252
     1  0.03227
     7  0.03175
     5  0.03102
     9  0.03055


Notice doc `3` and doc `1` — both top-3 in *both* rankers — come out on top of the fused list,
ahead of doc `9`, which was 4th in the vector ranking but never appears highly in the keyword
ranking. That's RRF's whole job: reward documents both signals agree on, without needing the two
signals' raw scores to be mutually comparable.

## 5. Endpoint sizing, limits, and where this fits the project's decision guide

| | Standard endpoint | Storage-optimized endpoint |
|---|---|---|
| Capacity | ~320M vectors @ 768 dims | > 1B vectors |
| Indexing speed | baseline | 10–20x faster |
| Query latency | lower | +~250ms |

This is precisely the **HNSW-vs-something-more-compressed** trade-off covered generally in
[`06_Choosing_and_Benchmarking.md`](../docs/Similarity_Search_Methods/06_Choosing_and_Benchmarking.md)
and [`05_Quantization_and_Compression.md`](../docs/Similarity_Search_Methods/05_Quantization_and_Compression.md):
past a certain scale, keeping every vector's full-precision HNSW graph in memory stops being
affordable, and the system has to compress — Databricks' storage-optimized tier is that same
trade-off, packaged as an endpoint sizing choice instead of an index-type choice you make yourself
with FAISS's `IndexIVFPQ`.

Other documented limits worth knowing: 50 indexes per endpoint, 500 endpoints per workspace, max
embedding dimension 4096, max 10,000 results per ANN query (200 for hybrid/full-text queries),
100KB max row size for Delta Sync indexes.

## Summary

- Databricks Vector Search is a **managed HNSW index** (L2 distance internally, cosine-equivalent
  when vectors are normalized — verified above) with automated Delta table sync and Unity Catalog
  governance layered on top.
- Its four index types differ in *who computes embeddings* and *how sync happens*, not in the
  underlying search algorithm.
- Hybrid search fuses dense (vector) and sparse (BM25 keyword) rankings via **Reciprocal Rank
  Fusion**, `k=60` — implemented and demonstrated above.
- Endpoint sizing (standard vs. storage-optimized) is the same recall/memory/speed trade-off this
  project's [IVF-PQ](ivf_pq.py) demonstrates generally, exposed as a managed-service knob instead
  of an index parameter you tune yourself.
- Every underlying concept here — HNSW, ANN vs. exact search, normalization for cosine-equivalence,
  the recall/memory/speed trade-off — is covered platform-agnostically in
  [`../docs/Similarity_Search_Methods/`](../docs/Similarity_Search_Methods/index.md); this notebook
  is where those concepts meet Databricks' specific terminology and defaults.